# Khởi tạo Môi trường và Cài đặt Thư viện
Cell này sẽ chuẩn bị mã nguồn từ đường dẫn Dataset và cài đặt các thư viện cần thiết.

In [ ]:
import os
import shutil

# 1. Định nghĩa đường dẫn dựa trên thông tin người dùng cung cấp
dataset_path = '/kaggle/input/datasets/nghahtrng/coffee-price-ie-source'
work_dir = '/kaggle/working/coffee-price-ie'

# 2. Xóa và tạo lại thư mục làm việc để đảm bảo sạch sẽ
if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
os.makedirs(work_dir, exist_ok=True)

print(f"🚀 Đang chuẩn bị mã nguồn từ: {dataset_path}")

# 3. Copy dữ liệu (Xử lý linh hoạt cấu trúc ZIP)
if os.path.exists(os.path.join(dataset_path, 'coffee-price-ie')):
    os.system(f'cp -r {dataset_path}/coffee-price-ie/* {work_dir}/')
else:
    os.system(f'cp -r {dataset_path}/* {work_dir}/')

os.chdir(work_dir)
print(f"✅ Setup thành công! Thư mục hiện tại: {os.getcwd()}")
print("Danh sách file:", os.listdir('.'))

# 4. Cài đặt thư viện (Chờ cho đến khi chạy xong, không bấm Stop)
print("\n📦 Đang cài đặt thư viện cần thiết...")
!pip install -q trafilatura bs4 underthesea huggingface_hub
!pip install -q transformers accelerate bitsandbytes
print("✅ Cài đặt thư viện hoàn tất!")

# Đăng nhập Hugging Face (Dành cho Llama 3.1)
Do Llama 3.1 là mô hình đóng (gated model), bạn cần đăng nhập bằng Access Token của Hugging Face.
Hãy thay `YOUR_HF_TOKEN` bằng token của bạn (lấy tại https://huggingface.co/settings/tokens).

In [ ]:
import os
from huggingface_hub import login

# Thử lấy token từ Kaggle Secrets (Add-ons -> Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception:
    # Nếu không có Secrets, bạn có thể dán thủ công vào đây (nhưng không khuyến khích)
    HF_TOKEN = "YOUR_HF_TOKEN"

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN":
    login(token=HF_TOKEN)
    print("✅ Đã cấu hình Hugging Face Token thành công!")
else:
    print("⚠️ CẢNH BÁO: Chưa cấu hình HF_TOKEN trong Kaggle Secrets. Llama 3.1 có thể không tải được.")

# 1. Bước 03: Parser (Xử lý HTML & Chặt câu)

In [ ]:
!python pipeline/03_parser/main.py
!ls -lh data/03_articles/

# 2. Bước 04: IE với Qwen2.5-7B
Sử dụng Qwen2.5-7B (4-bit quantization) để trích xuất. Hệ thống cũng sẽ tự động tạo bản đối chứng bằng Regex Baseline.

In [ ]:
!python pipeline/04_ie/main.py --model qwen
!ls -lh data/04_price_mentions/

# 3. Bước 04: IE với Llama-3.1-8B-Instruct
Chạy thêm Llama 3.1 để trích xuất. (Yêu cầu phải cấu hình HF Token ở trên).

In [ ]:
if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN":
    print("🚀 Bắt đầu chạy Llama 3.1...")
    !python pipeline/04_ie/main.py --model llama
    !ls -lh data/04_price_mentions/
else:
    print("⚠️ Bỏ qua Llama 3.1 vì chưa cấu hình Token hợp lệ.")

# 4. Bước 05: Disagreement Modeling (PhoBERT)
So sánh độ tương đồng giữa kết quả của Qwen và Llama 3.1 (nếu có), hoặc Qwen với Baseline bằng PhoBERT.

In [ ]:
!python pipeline/05_disagreement/main.py
!ls -lh data/05_disagreement/

# 5. Xuất kết quả
Nén lại tất cả dữ liệu kết quả để tải về máy.

In [ ]:
!zip -r /kaggle/working/extracted_data.zip data/03_articles/ data/04_price_mentions/ data/05_disagreement/
print("✅ All done! Bạn có thể tải file extracted_data.zip ở panel bên phải phần Output.")